# SUPERSEDED: generation now handled by `sdg/generate_runs.py`. Kept for provenance.

This notebook produced the single unseeded draw now at `synthetic_data/{method}_synthetic_LEGACY.csv`, which is what the pre-2026-08 fidelity/utility results were computed from. It is kept so those numbers stay traceable.

**Do not generate with this notebook.** `sdg/generate_runs.py` fits and samples the same plugin at the same hyperparameters, but seeded from `seeds.RUN_SEEDS` and repeated across runs, writing to `synthetic_data/runs/{method}_seed{seed}.csv`. The draws here cannot be reproduced (they were generated without a seed).


In [1]:
import os, time, tracemalloc
import pandas as pd
from pathlib import Path

METHOD = 'bayesian_network'
OUT_DIR = Path('../synthetic_data')
SDG_DIR = Path('.')
OUT_DIR.mkdir(exist_ok=True)

CONTINUOUS_COLS = ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
CATEGORICAL_COLS = ['workclass', 'marital_status', 'occupation', 'relationship',
                    'race', 'sex', 'native_country', 'income']
TARGET_COL = 'income'

In [2]:
df = pd.read_csv('../data/adult_train.csv')
df[CONTINUOUS_COLS] = df[CONTINUOUS_COLS].astype(float)
df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype('category')
N = len(df)
print(f'Training rows: {N}, columns: {df.columns.tolist()}')

Training rows: 21523, columns: ['age', 'workclass', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


In [3]:
try:
    from synthcity.plugins import Plugins
    from synthcity.plugins.core.dataloader import GenericDataLoader

    loader = GenericDataLoader(df, target_column=TARGET_COL)
    plugin = Plugins().get(METHOD)

    # --- fit ---
    tracemalloc.start()
    t0 = time.time()
    plugin.fit(loader)

    # --- generate ---
    synthetic = plugin.generate(count=N).dataframe()
    wall_time = round(time.time() - t0, 2)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    peak_mb = round(peak / 1e6, 2)

    synthetic.to_csv(OUT_DIR / f'{METHOD}_synthetic.csv', index=False)
    print(f'Done — {len(synthetic)} rows | {wall_time}s | {peak_mb} MB peak')
    print(synthetic.head())

    # --- overhead log ---
    overhead_path = SDG_DIR / 'computational_overhead.csv'
    row = pd.DataFrame([{'method': METHOD, 'rows_generated': len(synthetic),
                          'wall_time_s': wall_time, 'peak_memory_mb': peak_mb}])
    if overhead_path.exists():
        existing = pd.read_csv(overhead_path)
        existing = existing[existing['method'] != METHOD]
        row = pd.concat([existing, row], ignore_index=True)
    row.to_csv(overhead_path, index=False)

except Exception as e:
    import traceback
    log = f'ERROR running {METHOD}:\n{traceback.format_exc()}'
    print(log)
    (SDG_DIR / f'{METHOD}_logs.txt').write_text(log)

/opt/anaconda3/envs/priv-sdg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


[2026-06-27T17:29:19.599701+1000][70785][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-06-27T17:29:19.601725+1000][70785][CRITICAL] module disabled: /opt/anaconda3/envs/priv-sdg/lib/python3.10/site-packages/synthcity/plugins/generic/plugin_goggle.py
06/27/2026 17:29:32:WARNING:Probability values don't exactly sum to 1. Differ by: -2.220446049250313e-16. Adjusting values.
06/27/2026 17:29:32:WARNING:Probability values don't exactly sum to 1. Differ by: -2.220446049250313e-16. Adjusting values.
06/27/2026 17:29:32:WARNING:Probability values don't exactly sum to 1. Differ by: -2.220446049250313e-16. Adjusting values.
06/27/2026 17:29:32:WARNING:Probability values don't exactly sum to 1. Differ by: -2.220446049250313e-16. Adjusting values.
06/27/2026 17:29:32:WARNING:Probability values don't exactly sum to 1. Differ by: 1.1102230246251565e-16. Adjusting values.
06/27/2026 17:29:32:WARNING:Probability values don't exactly sum to 1. Differ by: -2.220446049250313e-16. A

Done — 21523 rows | 12.6s | 43.61 MB peak
         age workclass  education_num marital_status         occupation  \
0  50.014995   Private       8.984310       Divorced  Machine-op-inspct   
1  24.739453   Private       5.794558  Never-married              Sales   
2  29.004221   Private      13.000071  Never-married  Handlers-cleaners   
3  21.352912   Private       6.195277  Never-married      Other-service   
4  20.917390   Private      12.997596  Never-married       Craft-repair   

    relationship                race     sex  capital_gain  capital_loss  \
0  Not-in-family               White    Male   7395.001449      0.000000   
1      Own-child               Black  Female     32.055771      1.675403   
2  Not-in-family  Asian-Pac-Islander    Male      0.064364      0.003364   
3  Not-in-family  Amer-Indian-Eskimo  Female      9.266414      0.484311   
4  Not-in-family               White  Female      0.000000      0.000000   

   hours_per_week   native_country income  
0     